## Code For Zeroshot

In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import time

# ✅ Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
image_folder = "./RadSpineXR/Test/Annotated_150_images"
csv_path = "./RadSpineXR/Test/sample_150.csv"
output_csv = "./RadSpineXR/Test/BLIP2_zeroshot.csv"

# ✅ Create output directory if needed
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# ✅ Load processor and model
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16,
    device_map="auto"  # ✅ auto-distribution across GPUs
)


# ✅ Load the QA data
df = pd.read_csv(csv_path)
results = []

start = time.time()

# ✅ Process each image + question
for idx, row in df.iterrows():
    image_name = row["image_id"]
    question = row["question"]
    ground_truth = row.get("answer", "")

    image_path = os.path.join(image_folder, image_name)

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"⚠️ Error loading image {image_name}: {e}")
        continue

    try:
        # Preprocess and generate
        inputs = processor(images=image, text=question, return_tensors="pt").to(device, torch.float16)
        generated_ids = model.generate(**inputs)
        predicted_answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    except Exception as e:
        print(f"⚠️ Error during inference on image {image_name}: {e}")
        predicted_answer = "[ERROR]"

    # Save result
    results.append({
        "image_name": image_name,
        "question": question,
        "generated_answer": predicted_answer,
        "ground_truth": ground_truth
    })

    # Logging
    if idx % 10 == 0:
        print(f"[{idx}] Q: {question}\nA: {predicted_answer}\n")

    # Optional memory cleanup
    if torch.cuda.is_available() and idx % 10 == 0:
        torch.cuda.empty_cache()

    # Save partial progress every 50 rows
    #if idx % 50 == 0 and idx > 0:
       # partial_path = output_csv.replace(".csv", f"_partial_{idx}.csv")
        #pd.DataFrame(results).to_csv(partial_path, index=False)

# ✅ Save final results
output_df = pd.DataFrame(results)
output_df.to_csv(output_csv, index=False)

print(f"\n✅ Results saved to: {output_csv}")
print(f"🕒 Total time: {round(time.time() - start, 2)} seconds")


## Code For Finetuning

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from datasets import Dataset
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    TrainingArguments, Trainer, AutoTokenizer
)
import numpy as np
from torch.utils.data import DataLoader

# === Paths ===
csv_path = "./RadSpineXR/Train/sample_850.csv"
image_folder = "./RadSpineXR/Train/Unannotated_images"
output_dir = "./blip2-finetuned2-spine"

# === Load CSV and Dataset ===
df = pd.read_csv(csv_path)
print(f"Loaded dataset with {len(df)} examples")

# === Load Model and Processor ===
tokenizer = AutoTokenizer.from_pretrained("Salesforce/blip2-flan-t5-xl", use_fast=False)

# 2. Load processor and override its tokenizer
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
processor.tokenizer = tokenizer

# 3. Load the BLIP-2 model
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float32,
    device_map="auto"
)

# === Create a custom dataset class ===
class SpineXRDataset(torch.utils.data.Dataset):
    def __init__(self, df, processor, image_folder):
        self.df = df
        self.processor = processor
        self.image_folder = image_folder

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_folder, row["image_id"])
        
        try:
            image = Image.open(image_path).convert("RGB")
            question = row["question"]
            answer = row["answer"]
            
            # Process image and text inputs
            inputs = self.processor(images=image, text=question, return_tensors="pt", padding="max_length", max_length=512, truncation=True)
            
            # Process target text
            target = self.processor.tokenizer(answer, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
            
            # Remove batch dimension
            for k, v in inputs.items():
                inputs[k] = v.squeeze()
                
            labels = target["input_ids"].squeeze()
            # Replace padding token id with -100 so it's ignored in loss calculation
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            
            inputs["labels"] = labels
            
            # Ensure all tensors are float32, not float16
            if "pixel_values" in inputs and inputs["pixel_values"].dtype == torch.float16:
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.float32)
            
            return inputs
            
        except Exception as e:
            print(f"Error processing example {idx} (image: {row.get('image_id', 'unknown')}): {e}")
            # Return a dummy example that will be filtered out
            return None

# === Create train/validation split ===
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Training on {len(train_df)} examples, validating on {len(val_df)} examples")

# === Create datasets ===
train_dataset = SpineXRDataset(train_df, processor, image_folder)
val_dataset = SpineXRDataset(val_df, processor, image_folder)

# === Data Collator ===
def collate_fn(batch):
    # Filter out None values (failed examples)
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    
    # Prepare the batch
    pixel_values = torch.stack([example["pixel_values"] for example in batch])
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [example["input_ids"] for example in batch],
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [example["attention_mask"] for example in batch],
        batch_first=True,
        padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [example["labels"] for example in batch],
        batch_first=True,
        padding_value=-100
    )
    
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# === Custom Trainer with compute_metrics ===
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
#from rouge import Rouge

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Ignore num_items_in_batch if not needed
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only=False, ignore_keys=None):
        # Move inputs to appropriate device
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
            
        return (loss, outputs.logits, inputs["labels"])
    
    def compute_metrics(self, eval_preds):
        print("Computing metrics...")  # Debug print
        logits, labels = eval_preds
        
        # Convert logits to predictions
        preds = np.argmax(logits, axis=-1)
        
        # Decode predictions and labels
        pred_texts = []
        label_texts = []
        
        for pred, label in zip(preds, labels):
            # Filter out padding (-100)
            filtered_pred = [token for token in pred if token != -100]
            filtered_label = [token for token in label if token != -100]
            
            # Decode to text
            pred_text = processor.tokenizer.decode(filtered_pred, skip_special_tokens=True)
            label_text = processor.tokenizer.decode(filtered_label, skip_special_tokens=True)
            
            pred_texts.append(pred_text)
            label_texts.append(label_text)
        
        # Calculate metrics
        # Exact match accuracy
        exact_match = sum([1 if pred.strip() == label.strip() else 0 for pred, label in zip(pred_texts, label_texts)]) / len(pred_texts) if pred_texts else 0
        
        # BLEU score
        smoothie = SmoothingFunction().method4
        bleu_scores = []
        for pred, label in zip(pred_texts, label_texts):
            try:
                score = sentence_bleu([label.split()], pred.split(), smoothing_function=smoothie)
                bleu_scores.append(score)
            except:
                bleu_scores.append(0)
        
        avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
        
        # ROUGE score
        rouge = Rouge()
        rouge_scores = []
        for pred, label in zip(pred_texts, label_texts):
            try:
                score = rouge.get_scores(pred, label)[0]['rouge-l']['f']
                rouge_scores.append(score)
            except:
                rouge_scores.append(0)
        
        avg_rouge = sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0
        
        metrics = {
            "exact_match": exact_match,
            "bleu": avg_bleu,
            "rouge_l": avg_rouge  # Changed hyphen to underscore
        }
        
        print(f"Computed metrics: {metrics}")  # Debug print
        return metrics

# === Training Arguments ===
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",  # Use the full parameter name
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",  # Changed to use loss which is always available
    greater_is_better=False,  # Changed to False since we want to minimize loss
    report_to=None,  # Changed from None to enable proper reporting
    disable_tqdm=False,
    fp16=False,  # Disabled to avoid FP16 gradient issues
    bf16=False,  # Disabled unless you're sure your GPU supports it
    dataloader_drop_last=True,
    dataloader_num_workers=2,
    label_names=["labels"],  # Explicitly specify label names
)

# === Initialize Trainer ===
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

# === Start Training ===
print("🚀 Starting training...")
trainer.train()
print("✅ Training completed.")

# === Save the Fine-Tuned Model ===
trainer.save_model(output_dir)
print(f"💾 Model saved to '{output_dir}'")




## Inference Script 

In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# === Paths ===
test_csv = "./RadSpineXR/Test/sample_test.csv"
image_folder = "./RadSpineXR/Test/annotated images"
model_path = "./blip2-finetuned-spine"
output_csv = "./RadSpineXR/Test/predictions.csv"

# === Load test data ===
df = pd.read_csv(test_csv)

# === Load fine-tuned model and processor ===
processor = Blip2Processor.from_pretrained(model_path)
model = Blip2ForConditionalGeneration.from_pretrained(model_path)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# === Inference Loop ===
predictions = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    image_id = row["image_id"]
    question = row["question"]
    gt_answer = row["answer"]
    category = row["category"] if "category" in row else None

    try:
        image_path = os.path.join(image_folder, image_id)
        image = Image.open(image_path).convert("RGB")

        # Prepare inputs
        inputs = processor(images=image, text=question, return_tensors="pt").to(model.device)

        # Generate answer
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        pred_answer = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    except Exception as e:
        print(f"Failed on {image_id}: {e}")
        pred_answer = ""

    predictions.append({
        "image_id": image_id,
        "question": question,
        "ground_truth": gt_answer,
        "predicted_answer": pred_answer,
        "category": category
    })

# === Save predictions to CSV ===
pred_df = pd.DataFrame(predictions)
pred_df.to_csv(output_csv, index=False)

print(f"✅ Inference complete! Saved to: {output_csv}")
